# 02 — Feature Engineering Validation

Tests the expanded feature extraction pipeline (14 → 38 features) on the downloaded dataset.
Validates each feature group and checks for NaN/inf values.

**Input:** `data/labeled/` from notebook 01
**Output:** Feature statistics and validation results

In [ ]:
!pip install -q lightkurve astroquery PyWavelets matplotlib

import sys, os, glob

# Auto-detect codebase path — handles any Kaggle dataset structure
CODEBASE_PATH = None
for candidate in [
    '/kaggle/input/exopattern-codebase',
    *glob.glob('/kaggle/input/exopattern-codebase/*/'),
    *glob.glob('/kaggle/input/exopattern*/'),
    *glob.glob('/kaggle/input/exopattern*/*/'),
]:
    if os.path.isdir(os.path.join(candidate, 'backend')):
        CODEBASE_PATH = candidate.rstrip('/')
        break

if CODEBASE_PATH is None:
    raise FileNotFoundError(
        "Could not find 'backend/' directory in /kaggle/input/. "
        "Upload the project root (containing backend/ and experiments/) as a Kaggle dataset."
    )

sys.path.insert(0, CODEBASE_PATH)
print(f"Codebase found at: {CODEBASE_PATH}")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from backend.ml.preprocessing import LightCurvePreprocessor
from backend.ml.feature_names import FEATURE_GROUPS, get_feature_names, get_n_features

## 1. Feature Group Overview

In [ ]:
print("Feature Groups:")
total = 0
for group, names in FEATURE_GROUPS.items():
    print(f"  {group}: {len(names)} features")
    for n in names:
        print(f"    - {n}")
    total += len(names)
print(f"\nTotal: {total} features")

## 2. Extract Features from Dataset

In [ ]:
# Auto-detect data directory
DATA_DIR = None
for candidate in [
    '/kaggle/input/exopattern-labeled-data/data/labeled',
    *glob.glob('/kaggle/input/exopattern-labeled*/data/labeled'),
    *glob.glob('/kaggle/input/exopattern-labeled*/*/data/labeled'),
    '/kaggle/working/data/labeled',
    'data/labeled',
]:
    if os.path.exists(os.path.join(candidate, 'metadata.csv')):
        DATA_DIR = candidate
        break

if DATA_DIR is None:
    raise FileNotFoundError("Could not find labeled dataset (metadata.csv) in /kaggle/input/")

print(f"Using data from: {DATA_DIR}")

metadata = pd.read_csv(f'{DATA_DIR}/metadata.csv')
print(f"Loaded metadata for {len(metadata)} targets")

pp = LightCurvePreprocessor()
window_size = 50

In [ ]:
# Extract features for each group separately, then all together
row = metadata.iloc[0]
df = pd.read_csv(f"{DATA_DIR}/lightcurves/{row['filename']}")
df_proc = pp.preprocess(df, normalize=True)

print(f"Target: {row['target_id']} ({len(df_proc)} points after preprocessing)")
print()

groups = ['statistical', 'frequency', 'wavelet', 'autocorrelation', 'shape']
for group in groups:
    features = pp.extract_features(df_proc, window_size, feature_groups=[group])
    n_nan = np.isnan(features).sum()
    n_inf = np.isinf(features).sum()
    print(f"{group:20s}: shape={features.shape}, NaN={n_nan}, Inf={n_inf}")

# All features
features_all = pp.extract_features(df_proc, window_size)
print(f"{'ALL':20s}: shape={features_all.shape}, NaN={np.isnan(features_all).sum()}, Inf={np.isinf(features_all).sum()}")

## 3. Feature Statistics Across Dataset

In [ ]:
# Extract features from all targets
all_features = []
all_labels = []

for _, row in metadata.iterrows():
    lc_path = f"{DATA_DIR}/lightcurves/{row['filename']}"
    if not os.path.exists(lc_path):
        continue
    df = pd.read_csv(lc_path)
    labels = df['label'].values
    df_proc = pp.preprocess(df, normalize=True)
    features = pp.extract_features(df_proc, window_size)
    
    stride = max(1, window_size // 4)
    window_labels = []
    for i in range(0, len(df_proc) - window_size + 1, stride):
        window_lab = labels[i:i + window_size]
        window_labels.append(int(window_lab.max() > 0))
    
    n = min(len(features), len(window_labels))
    all_features.append(features[:n])
    all_labels.append(np.array(window_labels[:n]))

X = np.vstack(all_features)
y = np.concatenate(all_labels)

print(f"Total: {X.shape[0]} windows, {X.shape[1]} features")
print(f"Anomalous windows: {y.sum()} ({y.mean()*100:.1f}%)")
print(f"NaN count: {np.isnan(X).sum()}")
print(f"Inf count: {np.isinf(X).sum()}")

In [ ]:
# Feature statistics table
feature_names = get_feature_names()[:X.shape[1]]
stats_df = pd.DataFrame({
    'feature': feature_names,
    'mean': np.nanmean(X, axis=0),
    'std': np.nanstd(X, axis=0),
    'min': np.nanmin(X, axis=0),
    'max': np.nanmax(X, axis=0),
    'nan_count': np.isnan(X).sum(axis=0),
})
stats_df

## 4. Feature Distributions: Normal vs Transit Windows

In [ ]:
# Compare feature distributions between normal and transit windows
# Pick top 8 most discriminative features (by mean difference)
normal_means = np.nanmean(X[y == 0], axis=0)
transit_means = np.nanmean(X[y == 1], axis=0)
normal_stds = np.nanstd(X[y == 0], axis=0)

# Cohen's d effect size
effect_size = np.abs(transit_means - normal_means) / (normal_stds + 1e-10)
top_features = np.argsort(effect_size)[::-1][:8]

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for i, feat_idx in enumerate(top_features):
    ax = axes[i // 4, i % 4]
    ax.hist(X[y == 0, feat_idx], bins=50, alpha=0.5, label='Normal', density=True)
    ax.hist(X[y == 1, feat_idx], bins=50, alpha=0.5, label='Transit', density=True)
    ax.set_title(f"{feature_names[feat_idx]}\n(d={effect_size[feat_idx]:.2f})", fontsize=9)
    ax.legend(fontsize=7)

plt.suptitle('Top 8 Most Discriminative Features (by Cohen\'s d)', fontsize=12)
plt.tight_layout()
plt.savefig('/kaggle/working/feature_distributions.pdf', bbox_inches='tight')
plt.show()

## 5. Feature Correlation Matrix

In [ ]:
# Correlation matrix
corr = np.corrcoef(X.T)
corr = np.nan_to_num(corr)

fig, ax = plt.subplots(figsize=(12, 10))
im = ax.imshow(corr, cmap='RdBu_r', vmin=-1, vmax=1)
ax.set_xticks(range(len(feature_names)))
ax.set_yticks(range(len(feature_names)))
ax.set_xticklabels(feature_names, rotation=90, fontsize=6)
ax.set_yticklabels(feature_names, fontsize=6)
plt.colorbar(im)
ax.set_title('Feature Correlation Matrix')
plt.tight_layout()
plt.savefig('/kaggle/working/feature_correlations.pdf', bbox_inches='tight')
plt.show()

## Done!

Features look good. Proceed to notebook 03 for model training.